# TTS Experiment — Facebook MMS TTS + Coqui XTTS v2

**WeatherSpeak PH** — Gemma 4 Hackathon

## Objective

Evaluate Facebook's Massively Multilingual Speech (MMS) TTS models as a replacement
for Coqui XTTS v2. MMS provides **native** Cebuano and Tagalog models — no Spanish
phoneme approximation required.

### English TTS — what we tried

Three options were evaluated for English:

| Option | Model | Outcome |
|---|---|---|
| 1. MMS English | `facebook/mms-tts-eng` | ❌ Poor quality — robotic, unnatural prosody |
| 2. Microsoft SpeechT5 | `microsoft/speecht5_tts` + HiFiGAN | ❌ Scratchy output, inconsistent voice |
| 3. Coqui XTTS v2 | `tts_models/multilingual/multi-dataset/xtts_v2` | ✅ Best quality — retained for production |

### Final models used
| Language | Model | Type | Notes |
|---|---|---|---|
| Cebuano | `facebook/mms-tts-ceb` | VITS | Native phonemes, 36.3M params |
| Tagalog | `facebook/mms-tts-tgl` | VITS | Native phonemes, 36.3M params |
| English | `tts_models/multilingual/multi-dataset/xtts_v2` | Coqui XTTS v2 | Speaker: Damien Black |

### Bulletin
`PAGASA_20-19W_Pepito_SWB#01` — Severe Weather Bulletin, Tropical Depression Pepito

### Scope
All three languages (Cebuano, Tagalog, English). Missing TTS text files will be
auto-generated from markdown radio scripts using Gemma 4.

### License note
MMS TTS models are CC-BY-NC 4.0 (non-commercial). SpeechT5 is MIT licensed.
Coqui XTTS v2 is CPML licensed. All acceptable for this hackathon.
Flag MMS license for production review.


In [1]:
import re
import time
import numpy as np
import torch
import os
from pathlib import Path
from pydub import AudioSegment
from transformers import VitsModel, AutoTokenizer

# --- Paths ---
# Detect project root: look for pyproject.toml or CLAUDE.md
current_dir = Path.cwd()
print(f"DEBUG: Current working directory: {current_dir}")

# Walk up directory tree to find project root
project_root = current_dir
while project_root != project_root.parent:
    if (project_root / "pyproject.toml").exists() or (project_root / "CLAUDE.md").exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir

print(f"DEBUG: Detected project_root: {project_root}")

output_dir = project_root / "notebooks" / "08-mms-tts-experiment"
output_dir.mkdir(exist_ok=True, parents=True)
data_dir = project_root / "data"
input_dir = data_dir / "radio_bulletins"

# --- Experiment scope ---
STEM = "PAGASA_20-19W_Pepito_SWB#01"
# STEM = "PAGASA_25-TC22_Verbena_TCB#24"
# LANGUAGES = ["ceb", "tl", "en"]
LANGUAGES = ["ceb"]

# MMS models for Cebuano and Tagalog (native phonemes)
MMS_MODELS = {
    "ceb": "facebook/mms-tts-ceb",
    "tl":  "facebook/mms-tts-tgl",
}

# Coqui XTTS v2 config for English
XTTS_MODEL_NAME = "tts_models/multilingual/multi-dataset/xtts_v2"
XTTS_SPEAKER    = "Damien Black"
XTTS_SAMPLE_RATE = 24_000

# --- Check TTS text files ---
input_files = {lang: input_dir / f"{STEM}_tts_{lang}.txt" for lang in LANGUAGES}
existing    = {lang: p for lang, p in input_files.items() if p.exists()}
missing_tts = {lang: p for lang, p in input_files.items() if not p.exists()}

if existing:
    print(f"✓ Found {len(existing)} existing TTS text file(s):")
    for lang, p in existing.items():
        print(f"  {lang}: {p.name}")

if missing_tts:
    print(f"\n⚠  Missing {len(missing_tts)} TTS text file(s) — will generate in next cell:")
    for lang, p in missing_tts.items():
        print(f"  {lang}: {p.name}")

print(f"\n✓ Output dir: {output_dir.absolute()}")


/Users/josereyes/Dev/gemma4-hackathon/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEBUG: Current working directory: /Users/josereyes/Dev/gemma4-hackathon/notebooks
DEBUG: Detected project_root: /Users/josereyes/Dev/gemma4-hackathon
✓ Found 1 existing TTS text file(s):
  ceb: PAGASA_20-19W_Pepito_SWB#01_tts_ceb.txt

✓ Output dir: /Users/josereyes/Dev/gemma4-hackathon/notebooks/08-mms-tts-experiment


## 0A. Load & Format Bulletin Metadata

Instead of passing raw `ocr.md` to Gemma, we load the pre-extracted `metadata.json`
and format it into a structured, labelled text block.

**The problem with raw `ocr.md`:** The OCR output is noisy prose — Gemma sees
"maximum sustained winds of 120 km/h" and "movement speed" in the same unstructured
text and can hallucinate: mixing wind speed into the movement description.

**The fix:** `format_metadata_for_prompt()` produces clearly separated sections with
explicit labels (`INTENSITY`, `MOVEMENT`, `WIND SIGNALS IN EFFECT`) so there is no
ambiguity for the LLM.


In [2]:
import json

METADATA_PATH = (
    project_root / "data" / "gemma4_results" / "structured"
    / f"{STEM}_structured.json"
)

metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
print(f"✓ Loaded: {METADATA_PATH.name}")
print(f"  Storm    : {metadata['storm']['category']} {metadata['storm']['name']}")
print(f"  Position : {metadata['current_position']['reference']}")
print(f"  Winds    : {metadata['intensity']['max_sustained_winds_kph']} km/h")
print(f"  Movement : {metadata['movement']['direction']}, {metadata['movement'].get('speed_kph') or 'speed not specified'}")
active_signals = {k: v for k, v in metadata.get("affected_areas", {}).items() if v}
print(f"  Signals  : {'none' if not active_signals else active_signals}")


def format_metadata_for_prompt(metadata: dict) -> str:
    """Convert parsed metadata.json dict into a labelled text block for LLM prompts.

    Produces unambiguous field labels so the LLM cannot confuse wind speed with
    movement speed, and cannot miss the OUTSIDE PAR / no-signal status.
    """
    s = metadata.get("storm", {})
    storm_name = s.get("name", "Unknown")
    category   = s.get("category", "Unknown")
    intl       = s.get("international_name")
    intl_str   = f" (international name: {intl})" if intl else ""

    b_type     = metadata.get("bulletin_type", "")
    b_num      = metadata.get("bulletin_number")
    b_num_str  = f" #{b_num}" if b_num else ""
    bulletin_label = f"{b_type}{b_num_str}" if b_type else "Bulletin"

    iss        = metadata.get("issuance", {})
    issued     = iss.get("datetime", "not specified")
    valid_until = iss.get("valid_until") or "not specified"

    pos        = metadata.get("current_position", {})
    position_ref = pos.get("reference") or "not specified"
    position_as_of = pos.get("as_of") or ""
    position_str = position_ref + (f" (as of {position_as_of})" if position_as_of else "")

    inten      = metadata.get("intensity", {})
    winds      = inten.get("max_sustained_winds_kph")
    gusts      = inten.get("gusts_kph")
    winds_str  = f"{winds} km/h" if winds else "not specified"
    gusts_str  = f"up to {gusts} km/h" if gusts else "not specified"

    mov        = metadata.get("movement", {})
    direction  = mov.get("direction") or "not specified"
    speed      = mov.get("speed_kph")
    speed_str  = f"{speed} km/h" if speed else "not specified"

    areas = metadata.get("affected_areas", {})
    signal_sections = []
    for level in range(1, 6):
        places = areas.get(f"signal_{level}", [])
        if places:
            signal_sections.append(f"  Signal {level}: {', '.join(places)}")
    rainfall = areas.get("rainfall_warning", [])
    if rainfall:
        signal_sections.append(f"  Rainfall warning: {', '.join(rainfall)}")
    coastal = areas.get("coastal_waters")
    if coastal:
        signal_sections.append(f"  Coastal waters: {coastal}")

    signals_str = (
        "\n".join(signal_sections)
        if signal_sections
        else "  No wind signals in effect — no areas of the Philippines are under any wind signal."
    )

    forecasts = metadata.get("forecast_positions", [])
    forecast_lines = [
        f"  {fp.get('hour', '?')}-hour: {fp.get('reference') or 'location not specified'}"
        for fp in forecasts
    ]
    forecasts_str = "\n".join(forecast_lines) if forecast_lines else "  Not available"

    return (
        f"=== PAGASA TYPHOON BULLETIN ===\n"
        f"Storm: {category} {storm_name}{intl_str}\n"
        f"Bulletin: {bulletin_label}\n"
        f"Issued: {issued}\n"
        f"Valid until / Next bulletin: {valid_until}\n"
        f"\n"
        f"CURRENT POSITION:\n"
        f"  {position_str}\n"
        f"\n"
        f"INTENSITY:\n"
        f"  Maximum sustained winds: {winds_str} near the center\n"
        f"  Gusts: {gusts_str}\n"
        f"\n"
        f"MOVEMENT:\n"
        f"  Direction: {direction}\n"
        f"  Speed: {speed_str}\n"
        f"\n"
        f"WIND SIGNALS IN EFFECT:\n"
        f"{signals_str}\n"
        f"\n"
        f"FORECAST TRACK:\n"
        f"{forecasts_str}\n"
    )


bulletin_data = format_metadata_for_prompt(metadata)
print("\n" + "=" * 60)
print("Formatted bulletin data (LLM input):")
print("=" * 60)
print(bulletin_data)


✓ Loaded: PAGASA_20-19W_Pepito_SWB#01_structured.json


KeyError: 'storm'

## 0B. Generate Cebuano Radio Script from Structured Metadata

Use the structured `bulletin_data` (above) as the primary LLM input — replacing the noisy raw `ocr.md`.

**Known bugs in the current `radio_ceb.md` for Verbena TCB#24:**

| Bug | Cause |
|---|---|
| "Moving at 120 km/h" | LLM confused **wind speed** (120 km/h) with movement speed — raw OCR has no field labels |
| "Next advisory tomorrow afternoon" | TCB#24 is the **final bulletin** — storm has exited PAR |
| No mention of OUTSIDE PAR | Raw OCR buries this in prose; metadata makes it an explicit labelled field |

**Why the new approach works:** `format_metadata_for_prompt()` produces clearly-labelled sections (`INTENSITY`, `MOVEMENT`) and explicitly flags `OUTSIDE PAR` and `No wind signals in effect`. The LLM can no longer confuse the two numbers.

Set `FORCE_RADIO_REGEN = True` to regenerate even if the file already exists.


In [ ]:
import requests

# Set True to regenerate even if the file already exists on disk.
FORCE_RADIO_REGEN = True

RADIO_SCRIPT_PROMPTS = {
    "ceb": {
        "system": (
            "Ikaw nagsulat og mubo nga pahimangno bahin sa usa ka bagyo sa Cebuano "
            "nga ipakita sa usa ka website ug basahon sa makusog isip audio.\n\n"
            "KATUYOAN: Mabasa ug madungog kini sa mga Pilipino nga mahimong dili makasabot sa English — "
            "mga mag-uuma, mangingisda, ug mga komunidad nga kinahanglan mahibalo kung sila anaa sa peligro ug unsa ang buhaton. "
            "Ang matag pulong importante. Walay lugar alang sa bisan unsang dili makatulong kanila nga molihok.\n\n"
            "PAGKASUNOD SA IMPORMASYON — ibutang ang tanan niini, sa pagkasunod nga kini, sulod sa 200 ka pulong:\n"
            "  1. Ngalan sa bagyo ug kasamtangang kategorya (unsa kini)\n"
            "  2. Asa kini karon ug asa kini padulong — klaro ang lokasyon ug direksyon\n"
            "  3. Unsang mga lugar ang apektado ug unsang Signal level — o klaro nga wala'y signal (kinsa ang anaa sa peligro)\n"
            "  4. Unsa ang buhaton — paglikas, magpabilin sulod, likayi ang baybayon (aksyon)\n"
            "  5. Kanus-a ang sunod nga update — o kun kini ang katapusan nga bulletin\n\n"
            "ESTILO — PURO CEBUANO, WALAY ENGLISH:\n"
            "- Pagsulat sama sa imong gisulti sa imong silingan — simple, natural, dili komplikado\n"
            "- BAWAL ang mga pulong nga Ingles gawas sa mga pangalan sa tawo ug lugar (hal. Pepito, Catanduanes, Luzon)\n"
            "- Gamita ang inadlaw-adlaw nga Cebuano — dili pormal, dili opisyal, dili balita sa TV\n\n"
            "KRITIKAL NGA MGA LAGDA SA PAGBASA SA DATOS:\n"
            "- Ang seksyon nga INTENSITY mao ang KUSOG SA HANGIN (wind speed). "
            "Ang seksyon nga MOVEMENT mao ang DIREKSYON UG TULIN SA PAGBALHIN SA BAGYO. "
            "DILI kini managsama — AYAW ilaba ang duha.\n"
            "- Kun ang lokasyon nag-ingon OUTSIDE PAR, ang bagyo wala na sa Pilipinas. "
            "Klaro nga sultiha nga wala'y hulga sa atong mga lugar karon.\n"
            "- Kun walay Wind Signal, klaro nga sultiha nga walay lugar sa Pilipinas ang naapektuhan.\n"
            "- Ang 'Valid until / Next bulletin' nagpakita kun kanus-a ang sunod nga update. "
            "Kun 'not specified' ug taas na ang bulletin number, posible nga kini ang katapusan nga bulletin.\n\n"
            "FORMATTING: Natural nga daloy sa prosa. Walay headings, walay bullets, walay bold, walay markdown. "
            "Blank lines tali sa mga paragraph.\n\n"
            "GITAS-ON: Dili molapas sa 200 ka pulong."
        ),
        "user": (
            "I-convert kining structured nga datos sa PAGASA bulletin ngadto sa mubo nga pahimangno sa Cebuano.\n\n"
            "{bulletin_data}\n\n"
            "HINUMDUMI:\n"
            "- Basaha pag-ayo ang INTENSITY (wind speed) ug MOVEMENT (direksyon ug tulin sa pagbalhin sa bagyo). "
            "DILI kini managsama — AYAW ilaba.\n"
            "- Kun OUTSIDE PAR ug walay Signal, klaro nga sultiha nga wala'y hulga sa Pilipinas.\n"
            "- Puro Cebuano. Walay headings, walay markdown. Dili molapas sa 200 ka pulong."
        ),
    },
}


def generate_radio_script(bulletin_data: str, language: str) -> str:
    """Call Gemma 4 to generate a radio script from structured bulletin data."""
    p = RADIO_SCRIPT_PROMPTS[language]
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": p["system"]},
            {"role": "user",   "content": p["user"].format(bulletin_data=bulletin_data)},
        ],
        "stream": False,
    }
    resp = requests.post(f"{OLLAMA_API}/api/chat", json=payload, timeout=TIMEOUT)
    resp.raise_for_status()
    import re as _re
    content = resp.json()["message"]["content"].strip()
    content = _re.sub(r"<think>.*?</think>", "", content, flags=_re.DOTALL).strip()
    return content


radio_path = input_dir / f"{STEM}_radio_ceb.md"

if radio_path.exists() and not FORCE_RADIO_REGEN:
    print(f"✓ Radio script exists — skipping (set FORCE_RADIO_REGEN=True to regenerate)")
    radio_md = radio_path.read_text(encoding="utf-8")
else:
    print(f"Generating Cebuano radio script from metadata...")
    t_start = time.time()
    radio_md = generate_radio_script(bulletin_data, "ceb")
    elapsed = time.time() - t_start
    radio_path.write_text(radio_md, encoding="utf-8")
    print(f"✓ Generated in {elapsed:.1f}s  |  {len(radio_md.split())} words")
    print(f"✓ Saved → {radio_path.name}")

print(f"\n=== New Cebuano Radio Script ===\n")
print(radio_md)


In [ ]:
# # ── Before / After comparison ────────────────────────────────────────────────
# # Paste the old OCR-based script here so we can compare after regeneration.

# OLD_RADIO_CEB = """\
# Pagpahinumdom kining importante. Ang bagyo nga giilang VERBENA, kusog pa gihapon kaayo karon, apan anaa kini paglayo sa mga baybayon sa Palawan.

# Naglihok kini sa alas 120 kilometros sa usa ka oras, paingon sa kabugsanlan nga lawod sa Palawan, nga luwas pa sa atong pagpuyo.

# Sa pagkakaron, wala pa'y Signal nga gihayag sa bisan unsang dapit, apan nagpasabot ni nga kinahanglan gihapon kitang mag-andam. Bisan unsa pa man ka kalinaw ang atong makita, paglikay gihapon sa mga baybayon, ilabina kung kusog ang hangin o pagtaas sa balud.

# Kung adunay pag-ampo nga magpagawas og grabe nga balud o pag-ayo sa pag-uwan, kinahanglan gyud mag-andam ang mga pamilya nga mogawas o magpabilin sa sinul-oban nga lugar nga gisulti sa mga opisyales.

# Maghatag mi og sunod nga pahibalo pagkahapon sa hapon. Paminaw ug dili pasagdi ang mga instruksyon sa lokal nga kagamhanan. Andam mo, mga kaigsoonan.\
# """

# new_radio_path = input_dir / f"{STEM}_radio_ceb.md"
# new_radio_md = new_radio_path.read_text(encoding="utf-8") if new_radio_path.exists() else "(not yet generated)"

# print("=" * 60)
# print("BEFORE  (generated from raw ocr.md)")
# print("=" * 60)
# print(OLD_RADIO_CEB)

# print()
# print("=" * 60)
# print("AFTER   (generated from structured metadata.json)")
# print("=" * 60)
# print(new_radio_md)

# print()
# print("─── Known bugs in BEFORE ───────────────────────────────")
# print("✗  'Naglihok kini sa alas 120 kilometros sa usa ka oras'")
# print("   → 120 km/h is WIND SPEED (intensity), not movement speed")
# print("✗  'Maghatag mi og sunod nga pahibalo pagkahapon sa hapon'")
# print("   → TCB#24 is the final bulletin — no next advisory expected")
# print("✗  No mention of OUTSIDE PAR — storm has already left Philippine waters")


## 0. Generate Missing TTS Text Files

If any language is missing its TTS text file, generate it now using Gemma 4.
Reads from `_radio_{lang}.md`, generates dialect-pure plain text optimized for TTS.

**TTS text requirements:**
- Native language only (no English code-switching)
- English proper nouns spelled phonetically in target language
- No markdown formatting
- Paragraph breaks preserved (controls pause timing)

In [ ]:
import requests

# Languages to force-regenerate TTS text even if the file already exists.
# Set to ["ceb"] after regenerating the radio script in Section 0B — the TTS
# text must be rebuilt from the new script.
FORCE_REGEN = ["ceb"]

# Recompute missing after force-regen override
missing_tts_now = {
    lang: input_dir / f"{STEM}_tts_{lang}.txt"
    for lang in LANGUAGES
    if not (input_dir / f"{STEM}_tts_{lang}.txt").exists() or lang in FORCE_REGEN
}

if not missing_tts_now:
    print("✓ All TTS text files exist - skipping generation")
else:
    # Ollama API setup
    OLLAMA_API = "http://localhost:11434"
    MODEL_NAME = "gemma4:e4b"
    TIMEOUT = 10 * 60  # 10 minutes

    TTS_PROMPTS = {
        "en": {
            "system": (
                "You are converting a PAGASA severe weather announcement into plain text for text-to-speech synthesis.\n\n"
                "AUDIENCE: Filipinos with low literacy, limited education, and no English background. "
                "Keep the language simple. Short sentences. Common words only.\n\n"
                "RULES:\n"
                "- NO markdown: no headings (#), no bullet points (-), no asterisks (*), no bold/italic\n"
                "- NO placeholders. Never write [station name], [insert...], [your location], or anything in brackets.\n"
                "- NO radio show language. No 'Good morning listeners', no sign-offs, no station IDs.\n"
                "- Rewrite as natural flowing prose — paragraph breaks (blank lines) for pausing\n"
                "- Use simple, short words. If the original uses a complex word, use a simpler one.\n"
                "- DO NOT add any information that was not in the original script\n"
                "- Output: plain text only, no markup or formatting characters"
            ),
            "user": (
                "Read this markdown weather announcement and rewrite it as TTS-ready plain English text.\n\n"
                "{markdown}\n\n"
                "Write the plain English text now. Simple words. Short sentences. "
                "Paragraph breaks (blank lines) for natural pausing. No markdown. No placeholders."
            ),
        },
        "tl": {
            "system": (
                "Ikaw ay nagko-convert ng PAGASA severe weather announcement sa plain text para sa text-to-speech synthesis.\n\n"
                "AUDIENCE: Mga Pilipinong may mababang literacy, limitadong edukasyon, at walang English background. "
                "Gumamit ng simple na wika. Maikling mga pangungusap. Mga karaniwang salita lamang.\n\n"
                "PINAKAMAHALAGANG PANUNTUNAN:\n"
                "WALANG INGLES. BAWAT salitang Ingles na makikita mo sa script ay DAPAT palitan ng Tagalog o ng phonetically spelled na anyo. "
                "Ang tanging pagbubukod ay mga pangalan ng tao at lugar (hal. 'Pepito', 'Catanduanes', 'Isabela').\n\n"
                "WALANG PLACEHOLDER. Huwag isulat ang [pangalan ng istasyon], [ilagay...], [iyong lokasyon], o anumang nasa brackets.\n\n"
                "WALANG RADIO SHOW NA WIKA. Walang 'Magandang umaga mga tagapakinig', walang sign-offs, walang station IDs.\n\n"
                "MANDATORY NA PHONETIC SPELLINGS — gamitin ang mga ito palagi, hindi ang Ingles:\n"
                "  - Tropical Depression → tro-pi-kal di-pre-syon\n"
                "  - Tropical Storm → tro-pi-kal storm\n"
                "  - Severe Tropical Storm → se-beer tro-pi-kal storm\n"
                "  - Typhoon → tai-pun\n"
                "  - Super Typhoon → su-per tai-pun\n"
                "  - PAGASA / PAG-ASA → pag-asa\n"
                "  - forecast → pore-kast\n"
                "  - advisory → ad-bay-so-ri\n"
                "  - bulletin → bu-le-tin\n"
                "  - warning → wor-ning\n"
                "  - update → ap-deyt\n"
                "  - Signal Number One / Two / Three / Four / Five → sig-nal nam-ber wan / tu / tri / por / payb\n"
                "  - kilometers per hour / kph / km/h → ki-lo-me-tro ba-wat o-ras\n"
                "  - northeast / southeast / northwest / southwest → nor-ist / sow-ist / nor-west / sow-west\n"
                "  - north / south / east → nor / sow / ist\n"
                "  - northern / southern / eastern / western → nor-dern / sow-dern / is-tern / wes-tern\n"
                "  - Low Pressure Area / LPA → mababang presyon\n"
                "PARA SA MGA NUMERO — gamita ang Filipino/Spanish na mga salita:\n"
                "  - 25 km/h → beinte singko ki-lo-me-tro ba-wat o-ras\n"
                "  - 65 km/h → sisenta y singko ki-lo-me-tro ba-wat o-ras\n"
                "  - 95 km/h → nobenta y singko ki-lo-me-tro ba-wat o-ras\n"
                "  - 120 km/h → isang daan at dalawampu ki-lo-me-tro ba-wat o-ras\n"
                "  - 130 km/h → isang daan at tatlumpu ki-lo-me-tro ba-wat o-ras\n"
                "  - 150 km/h → isang daan at limampu ki-lo-me-tro ba-wat o-ras\n"
                "  - 200 km/h → dalawang daan ki-lo-me-tro ba-wat o-ras\n"
                "  - Para sa iba pang numero: 5=singko, 10=diyes, 15=kinse, 20=beinte,\n"
                "    30=treynta, 40=kuwarenta, 50=singkwenta, 60=sisenta,\n"
                "    70=sitenta, 80=otsenta, 90=nobenta, 100=isang daan\n\n"
                "  - hPa → ek-to-pas-kal\n"
                "  - coastal → kos-tal\n"
                "  - landfall → land-pol\n"
                "  - storm surge → storm serj\n"
                "  - flash flood → plash plud\n"
                "  - emergency → i-mer-chen-si\n"
                "  - evacuation → i-bak-yu-ey-syon\n"
                "  - center → sen-ter\n"
                "  - official → o-pi-syal\n"
                "  - Luzon → lu-son\n"
                "  - Visayas → bi-sa-yas\n"
                "  - Mindanao → min-da-naw\n\n"
                "IBA PANG PANUNTUNAN:\n"
                "- WALANG markdown: walang # headings, walang - bullets, walang * bold/italic\n"
                "- Isulat bilang natural na daloy ng prosa na angkop para basahin nang malakas\n"
                "- Panatilihin ang paragraph structure: blank lines sa pagitan ng mga paragraph\n"
                "- HUWAG magdagdag ng anumang texto na wala sa orihinal na script\n"
                "- Output: plain text lamang"
            ),
            "user": (
                "Basahin ang markdown weather announcement na ito at isulat muli ito bilang TTS-ready plain Tagalog text.\n\n"
                "{markdown}\n\n"
                "TANDAAN: Tagalog lamang — WALANG INGLES maliban sa mga pangalan ng tao at lugar. "
                "Gamitin ang phonetically spelled na anyo para sa lahat ng teknikal na termino. "
                "Walang placeholder. Walang radio show na wika. "
                "Paragraph breaks (blank lines) para sa natural na pausing. Walang markdown."
            ),
        },
        "ceb": {
            "system": (
                "Ikaw kay mag-convert sa PAGASA severe weather announcement ngadto sa plain text para sa text-to-speech synthesis.\n\n"
                "AUDIENCE: Mga Pilipino nga di kaayo literate, limitado ang edukasyon, ug gamay ra ang English background. "
                "Gamita ang simple nga pinulongan. Mubo nga mga sentence. Komon nga mga pulong lamang.\n\n"
                "PINAKA-IMPORTANTE NGA LAGDA:\n"
                "WALAY ENGLISH. ANG MATAG English word nga imong makita sa script KINAHANGLAN ilisdan sa Cebuano o sa phonetically spelled nga porma. "
                "Ang bugtong eksepsyon mao ang mga pangalan sa tawo ug lugar (sampol. 'Pepito', 'Catanduanes', 'Isabela').\n\n"
                "ug ang mga Proper Nouns nga dili ma-translate (sampol. 'PAGASA', 'Signal Number One').\n\n"
                "WALAY PLACEHOLDER. Ayaw isulat ang [ngalan sa istasyon], [ibutang...], [imong lokasyon], o bisan unsa nga anaa sa brackets.\n\n"
                "WALAY RADIO SHOW NGA PULONG. Walay 'Maayong buntag mga tigpaminaw', walay sign-offs, walay station IDs.\n\n"
                "MANDATORY NGA PHONETIC SPELLINGS — gamita kini kanunay, dili ang English:\n"
                "  - Tropical Depression → tro-pi-kal di-pre-syon\n"
                "  - Tropical Storm → tro-pi-kal storm\n"
                "  - Severe Tropical Storm → se-beer tro-pi-kal storm\n"
                "  - Typhoon → tai-pun\n"
                "  - Super Typhoon → su-per tai-pun\n"
                "  - PAGASA / PAG-ASA → pag-asa\n"
                "  - forecast → pore-kast\n"
                "  - advisory → ad-bay-so-ri\n"
                "  - bulletin → bu-le-tin\n"
                "  - warning → wor-ning\n"
                "  - update → ap-deyt\n"
                "  - Signal Number One / Two / Three / Four / Five → sig-nal nam-ber wan / tu / tri / por / payb\n"
                "  - kilometers per hour / kph / km/h → ki-lo-me-tros sa usa ka oras\n"
                "  - northeast / southeast / northwest / southwest → nor-ist / sow-ist / nor-west / sow-west\n"
                "  - north / south / east → nor / sow / ist\n"
                "  - northern / southern / eastern / western → nor-dern / sow-dern / is-tern / wes-tern\n"
                "  - Low Pressure Area / LPA → ubos nga presyon\n"
                "PARA SA MGA NUMERO — gamita ang Cebuano/Spanish nga mga pulong:\n"
                "  - 25 km/h → baynte singko ki-lo-me-tros sa usa ka oras\n"
                "  - 65 km/h → sisenta y singko ki-lo-me-tros sa usa ka oras\n"
                "  - 95 km/h → nobenta y singko ki-lo-me-tros sa usa ka oras\n"
                "  - 120 km/h → isyento baynte ki-lo-me-tros sa usa ka oras\n"
                "  - 130 km/h → isyento treynta ki-lo-me-tros sa usa ka oras\n"
                "  - 150 km/h → isyento singkwenta ki-lo-me-tros sa usa ka oras\n"
                "  - 200 km/h → dos siyentos ki-lo-me-tros sa usa ka oras\n"
                "  - Para sa ubang numero: 5=singko, 10=diyes, 15=kinse, 20=baynte,\n"
                "    30=treynta, 40=kuwarenta, 50=singkwenta, 60=sisenta,\n"
                "    70=sitenta, 80=otsenta, 90=nobenta, 100=isyento\n\n"
                "  - hPa → ek-to-pas-kal\n"
                "  - coastal → kos-tal\n"
                "  - landfall → land-pol\n"
                "  - storm surge → storm serj\n"
                "  - flash flood → plash plud\n"
                "  - emergency → i-mer-chen-si\n"
                "  - evacuation → i-bak-yu-ey-syon\n"
                "  - center → sen-ter\n"
                "  - official → o-pi-syal\n"
                "  - Luzon → lu-son\n"
                "  - Visayas → bi-sa-yas\n"
                "  - Mindanao → min-da-naw\n\n"
                "UBAN PA NGA MGA LAGDA:\n"
                "- WALAY markdown: walay # headings, walay - bullets, walay * bold/italic\n"
                "- Isulat isip natural nga daloy sa prosa nga angay basahon sa makusog\n"
                "- Pahimusa ang paragraph structure: blank lines tali sa mga paragraph\n"
                "- AYAW pagdugang og bisan unsa nga texto nga wala sa orihinal nga script\n"
                "- Output: plain text lamang"
            ),
            "user": (
                "Basaha kining markdown weather announcement ug isulat kini pag-usab isip TTS-ready plain Cebuano text.\n\n"
                "{markdown}\n\n"
                "HINUMDUMI: Cebuano lamang — WALAY ENGLISH gawas sa mga pangalan sa tawo ug lugar. "
                "Gamita ang phonetically spelled nga porma para sa tanan nga teknikal nga termino. "
                "Walay placeholder. Walay radio show nga pulong. "
                "Paragraph breaks (blank lines) para sa natural nga pausing. Walay markdown."
            ),
        },
    }

    def generate_tts_text(stem: str, language: str) -> None:
        """Generate TTS text file from markdown radio script using Gemma 4."""
        lang_names = {"ceb": "Cebuano", "tl": "Tagalog", "en": "English"}
        print(f"\nGenerating TTS text: {stem} ({lang_names[language]})...")

        radio_md_path = input_dir / f"{stem}_radio_{language}.md"
        if not radio_md_path.exists():
            raise FileNotFoundError(f"Radio markdown not found: {radio_md_path}")

        markdown_script = radio_md_path.read_text(encoding="utf-8")

        payload = {
            "model": MODEL_NAME,
            "messages": [
                {"role": "system", "content": TTS_PROMPTS[language]["system"]},
                {"role": "user", "content": TTS_PROMPTS[language]["user"].format(markdown=markdown_script)},
            ],
            "stream": False,
        }

        t_start = time.time()
        response = requests.post(f"{OLLAMA_API}/api/chat", json=payload, timeout=TIMEOUT)
        response.raise_for_status()
        elapsed = time.time() - t_start

        tts_text = response.json().get("message", {}).get("content", "").strip()

        out_path = input_dir / f"{stem}_tts_{language}.txt"
        out_path.write_text(tts_text, encoding="utf-8")

        word_count = len(tts_text.split())
        print(f"  ✓ Generated in {elapsed:.1f}s")
        print(f"  Words: {word_count}")
        print(f"  Saved → {out_path.name}")

    for lang in missing_tts_now.keys():
        generate_tts_text(STEM, lang)

    input_files = {lang: input_dir / f"{STEM}_tts_{lang}.txt" for lang in LANGUAGES}
    print(f"\n✓ All TTS text files ready ({len(LANGUAGES)} languages)")


In [ ]:

# Second Gemma 4 pass — detect and replace any English words that slipped through
# This runs on the generated TTS text files and rewrites them in-place.

CLEANUP_PROMPTS = {
    "tl": {
        "system": (
            "Ikaw ay isang editor ng Tagalog TTS text. Ang iyong trabaho ay hanapin ang mga salitang Ingles "
            "at palitan ang mga ito ng tamang Tagalog o phonetically spelled na anyo.\n\n"
            "PANUNTUNAN:\n"
            "- Hanapin ang LAHAT ng salitang Ingles sa text\n"
            "- Palitan ng Tagalog na katumbas o phonetically spelled na anyo (gamit ang mga gitling)\n"
            "- Mga pangalan ng tao at lugar ay HINDI dapat palitan (hal. Pepito, Catanduanes, Luzon)\n"
            "- Huwag baguhin ang anumang bagay na hindi Ingles\n"
            "- Ibalik ang BUONG text na may mga pagbabago lamang\n"
            "- Walang markdown, walang paliwanag — plain text lamang\n\n"
            "HALIMBAWA NG PAGPAPALIT:\n"
            "  'storm surge' → 'storm serj'\n"
            "  'landfall' → 'land-pol'\n"
            "  'coastal' → 'kos-tal'\n"
            "  'warning' → 'wor-ning'\n"
            "  'advisory' → 'ad-bay-so-ri'\n"
            "  'signal' → 'sig-nal'\n"
            "  'forecast' → 'pore-kast'\n"
            "  'emergency' → 'i-mer-chen-si'\n"
            "  'evacuation' → 'i-bak-yu-ey-syon'"
        ),
        "user": (
            "Suriin ang Tagalog TTS text na ito. Hanapin ang lahat ng salitang Ingles at palitan ng "
            "Tagalog o phonetically spelled na anyo. Ibalik ang buong text na may mga pagbabago.\n\n"
            "{text}"
        ),
    },
    "ceb": {
        "system": (
            "Ikaw usa ka editor sa Cebuano TTS text. Ang imong trabaho mao ang pangitaon ang mga pulong nga Ingles "
            "ug ilisan kini sa husto nga Cebuano o phonetically spelled nga porma.\n\n"
            "MGA LAGDA:\n"
            "- Pangitaa ang TANAN nga pulong nga Ingles sa text\n"
            "- Ilisan sa Cebuano nga katumbas o phonetically spelled nga porma (gamit ang mga gitling)\n"
            "- Ang mga pangalan sa tawo ug lugar DILI isulat pag-usab (hal. Pepito, Catanduanes, Luzon)\n"
            "- Ayaw usba ang bisan unsang butang nga dili Ingles\n"
            "- Ibalik ang TIBUOK text nga adunay mga pagbabago lamang\n"
            "- Walay markdown, walay paliwanag — plain text lamang\n\n"
            "PANANGLITAN SA PAGPULI:\n"
            "  'storm surge' → 'storm serj'\n"
            "  'landfall' → 'land-pol'\n"
            "  'coastal' → 'kos-tal'\n"
            "  'warning' → 'wor-ning'\n"
            "  'advisory' → 'ad-bay-so-ri'\n"
            "  'signal' → 'sig-nal'\n"
            "  'forecast' → 'pore-kast'\n"
            "  'emergency' → 'i-mer-chen-si'\n"
            "  'evacuation' → 'i-bak-yu-ey-syon'\n"
            "  'mo-intensify' → 'mo-kusog'\n"
        ),
        "user": (
            "Susiha kining Cebuano TTS text. Pangitaa ang tanan nga pulong nga Ingles ug ilisan sa "
            "Cebuano o phonetically spelled nga porma. Ibalik ang tibuok text nga adunay mga pagbabago.\n\n"
            "{text}"
        ),
    },
}


def cleanup_english_words(stem: str, language: str) -> None:
    """Second Gemma 4 pass: replace any remaining English words in a TTS text file."""
    if language not in CLEANUP_PROMPTS:
        return  # English needs no cleanup

    tts_path = input_dir / f"{stem}_tts_{language}.txt"
    if not tts_path.exists():
        print(f"  ⚠ Skipping {language} — file not found: {tts_path.name}")
        return

    original_text = tts_path.read_text(encoding="utf-8")

    lang_label = {"tl": "Tagalog", "ceb": "Cebuano"}[language]
    print(f"  Cleaning {lang_label} TTS text...")

    OLLAMA_API = "http://localhost:11434"
    MODEL_NAME = "gemma4:e4b"
    TIMEOUT = 10 * 60  # 10 minutes
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": CLEANUP_PROMPTS[language]["system"]},
            {"role": "user", "content": CLEANUP_PROMPTS[language]["user"].format(text=original_text)},
        ],
        "stream": False,
    }

    t_start = time.time()
    response = requests.post(f"{OLLAMA_API}/api/chat", json=payload, timeout=TIMEOUT)
    response.raise_for_status()
    elapsed = time.time() - t_start

    cleaned_text = response.json().get("message", {}).get("content", "").strip()
    tts_path.write_text(cleaned_text, encoding="utf-8")

    print(f"  ✓ {lang_label} cleaned in {elapsed:.1f}s ({len(original_text)} → {len(cleaned_text)} chars)")


print("Running English-word cleanup pass (TL + CEB)...")
for lang in ["tl", "ceb"]:
    cleanup_english_words(STEM, lang)
print("✓ Cleanup complete")

In [ ]:
# Third Gemma 4 pass — convert all remaining digits to spoken words
# Runs after the English cleanup so numbers like "21" become "baynte uno", etc.

NUMBER_CLEANUP_PROMPTS = {
    "tl": {
        "system": (
            "Ikaw ay isang editor ng Tagalog TTS text. Ang iyong trabaho ay hanapin ang LAHAT ng numerong nakasulat "
            "bilang mga digit at palitan sila ng katumbas na salita sa Filipino/Spanish na sistema ng bilang.\n\n"
            "PANUNTUNAN:\n"
            "- Hanapin ang BAWAT numero na nakasulat bilang digit (0-9) sa text\n"
            "- Palitan ng spoken na anyo gamit ang Filipino/Spanish na mga salita\n"
            "- Ang mga pangalan ng tao at lugar ay HUWAG baguhin (hal. Signal 2 → sig-nal tu)\n"
            "- Huwag baguhin ang anumang salita — digits lang ang palitan\n"
            "- Ibalik ang BUONG text na may mga pagbabago lamang\n"
            "- Walang markdown, walang paliwanag — plain text lamang\n\n"
            "MGA NUMERO AT KATUMBAS:\n"
            "  1=uno  2=dos  3=tres  4=kuwatro  5=singko\n"
            "  6=sayis  7=syete  8=otso  9=nuwebe  10=diyes\n"
            "  11=onse  12=dose  13=trese  14=katorse  15=kinse\n"
            "  16=disisayis  17=disisyete  18=diotso  19=disinuwebe\n"
            "  20=beinte  21=beinte uno  22=beinte dos  25=beinte singko\n"
            "  30=treynta  31=treynta y uno  40=kuwarenta  50=singkwenta\n"
            "  60=sisenta  70=sitenta  80=otsenta  90=nobenta\n"
            "  100=isang daan  120=isang daan beinte  130=isang daan treynta\n"
            "  150=isang daan singkwenta  200=dos siyentos\n\n"
            "HALIMBAWA:\n"
            "  'Oktubre 21' → 'Oktubre beinte uno'\n"
            "  '25 kilometro' → 'beinte singko kilometro'\n"
            "  '130 kilometro' → 'isang daan treynta kilometro'\n"
            "  'Signal 2' → 'sig-nal tu'\n"
            "  '6 ng umaga' → 'sayis ng umaga'"
        ),
        "user": (
            "Suriin ang Tagalog TTS text na ito. Palitan ang LAHAT ng digit na numero ng katumbas na salita. "
            "Ibalik ang buong text na may mga pagbabago.\n\n"
            "{text}"
        ),
    },
    "ceb": {
        "system": (
            "Ikaw usa ka editor sa Cebuano TTS text. Ang imong trabaho mao ang pangitaon ang TANAN nga numero nga "
            "gisulat isip mga digit ug ilisan kini sa katumbas nga pulong sa Cebuano/Spanish nga sistema sa ihap.\n\n"
            "MGA LAGDA:\n"
            "- Pangitaa ang MATAG numero nga gisulat isip digit (0-9) sa text\n"
            "- Ilisan sa spoken nga porma gamit ang Cebuano/Spanish nga mga pulong\n"
            "- Ang mga pangalan sa tawo ug lugar DILI usbon (hal. Signal 2 → sig-nal tu)\n"
            "- Ayaw usba ang bisan unsang pulong — ang mga digit lang ang ilisan\n"
            "- Ibalik ang TIBUOK text nga adunay mga pagbabago lamang\n"
            "- Walay markdown, walay paliwanag — plain text lamang\n\n"
            "MGA NUMERO UG KATUMBAS:\n"
            "  1=uno  2=dos  3=tres  4=kuwatro  5=singko\n"
            "  6=sayis  7=syete  8=otso  9=nuwebe  10=diyes\n"
            "  11=onse  12=dose  13=trese  14=katorse  15=kinse\n"
            "  16=disisayis  17=disisyete  18=diotso  19=disinuwebe\n"
            "  20=baynte  21=baynte uno  22=baynte dos  25=baynte singko\n"
            "  30=treynta  31=treynta y uno  40=kuwarenta  50=singkwenta\n"
            "  60=sisenta  70=sitenta  80=otsenta  90=nobenta\n"
            "  100=isyento  120=isyento baynte  130=isyento treynta\n"
            "  150=isyento singkwenta  200=dos siyentos\n\n"
            "PANANGLITAN:\n"
            "  'Oktubre 21' → 'Oktubre baynte uno'\n"
            "  '25 kilometros' → 'baynte singko kilometros'\n"
            "  '130 kilometros' → 'isyento treynta kilometros'\n"
            "  'Signal 2' → 'sig-nal tu'\n"
            "  '6 sa buntag' → 'sayis sa buntag'"
        ),
        "user": (
            "Susiha kining Cebuano TTS text. Ilisan ang TANAN nga digit nga numero sa katumbas nga pulong. "
            "Ibalik ang tibuok text nga adunay mga pagbabago.\n\n"
            "{text}"
        ),
    },
}


def cleanup_numbers(stem: str, language: str) -> None:
    """Third Gemma 4 pass: convert all digit numbers to spoken words in the target language."""
    if language not in NUMBER_CLEANUP_PROMPTS:
        return  # English TTS handles numbers fine

    tts_path = input_dir / f"{stem}_tts_{language}.txt"
    if not tts_path.exists():
        print(f"  ⚠ Skipping {language} — file not found: {tts_path.name}")
        return

    original_text = tts_path.read_text(encoding="utf-8")

    lang_label = {"tl": "Tagalog", "ceb": "Cebuano"}[language]
    print(f"  Converting numbers in {lang_label} TTS text...")

    OLLAMA_API = "http://localhost:11434"
    MODEL_NAME = "gemma4:e4b"
    TIMEOUT = 10 * 60  # 10 minutes
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": NUMBER_CLEANUP_PROMPTS[language]["system"]},
            {"role": "user", "content": NUMBER_CLEANUP_PROMPTS[language]["user"].format(text=original_text)},
        ],
        "stream": False,
    }

    t_start = time.time()
    response = requests.post(f"{OLLAMA_API}/api/chat", json=payload, timeout=TIMEOUT)
    response.raise_for_status()
    elapsed = time.time() - t_start

    cleaned_text = response.json().get("message", {}).get("content", "").strip()
    tts_path.write_text(cleaned_text, encoding="utf-8")

    print(f"  ✓ {lang_label} numbers converted in {elapsed:.1f}s ({len(original_text)} → {len(cleaned_text)} chars)")


print("Running number conversion pass (TL + CEB)...")
for lang in ["tl", "ceb"]:
    cleanup_numbers(STEM, lang)
print("✓ Number conversion complete")

## 1. Prepare Sentences for Synthesis

Split the TTS plain text into individual sentences with paragraph boundary flags.

**Processing differs by language:**
- **Cebuano/Tagalog (VITS):** Lowercase, punctuation-stripped (MMS TTS requirements)
- **English (Coqui XTTS v2):** Preserves capitalisation and punctuation for natural prosody

**Pause timings:** 500ms between sentences, 750ms between paragraphs.

Input: `data/radio_bulletins/{stem}_tts_{lang}.txt`

In [ ]:
import re


def prepare_mms_sentences(text: str) -> list[tuple[str, bool]]:
    """Split plain text into MMS-ready sentences with paragraph boundary flags.
    
    Used for VITS models (Cebuano, Tagalog) which require lowercase, unpunctuated text.

    Returns list of (sentence, is_paragraph_end) tuples where:
    - sentence: lowercase, punctuation-stripped (in-word apostrophes/hyphens preserved)
    - is_paragraph_end: True if last sentence of its paragraph (triggers longer pause)
    """
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    result = []
    for paragraph in paragraphs:
        sentences = re.split(r'(?<=[.!?])\s+', paragraph)
        sentences = [s.strip() for s in sentences if s.strip()]
        for sent_idx, sentence in enumerate(sentences):
            is_last_in_para = (sent_idx == len(sentences) - 1)
            sentence = sentence.lower()
            # Remove all punctuation except apostrophes and hyphens
            s = re.sub(r"[^\w\s'\-]", " ", sentence)
            # Remove apostrophes/hyphens not flanked by word characters (standalone)
            s = re.sub(r"(?<!\w)['\-]|['\-](?!\w)", " ", s)
            sentence = re.sub(r"\s+", " ", s).strip()
            if sentence:
                result.append((sentence, is_last_in_para))
    return result


def prepare_english_sentences(text: str) -> list[tuple[str, bool]]:
    """Split plain text into Coqui XTTS v2-ready sentences with paragraph boundary flags.

    Preserves capitalisation and punctuation for natural English prosody (XTTS v2).

    Returns list of (sentence, is_paragraph_end) tuples where:
    - sentence: preserves original capitalization and punctuation
    - is_paragraph_end: True if last sentence of its paragraph (triggers longer pause)
    """
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    result = []
    for paragraph in paragraphs:
        sentences = re.split(r'(?<=[.!?])\s+', paragraph)
        sentences = [s.strip() for s in sentences if s.strip()]
        for sent_idx, sentence in enumerate(sentences):
            is_last_in_para = (sent_idx == len(sentences) - 1)
            # Normalize whitespace but preserve everything else
            sentence = re.sub(r"\s+", " ", sentence).strip()
            if sentence:
                result.append((sentence, is_last_in_para))
    return result


# Prepare sentences for all scripts
sentence_lists = {}
for lang in LANGUAGES:
    text = input_files[lang].read_text(encoding="utf-8")
    # Use appropriate preparation for each language
    if lang == "en":
        sentences = prepare_english_sentences(text)
    else:
        sentences = prepare_mms_sentences(text)
    sentence_lists[lang] = sentences
    print(f"✓ {lang}: {len(sentences)} sentences from {len(text)} chars")
    print(f"  First: {sentences[0][0][:80]!r}  (para_end={sentences[0][1]})")
    print(f"  Last:  {sentences[-1][0][:80]!r}  (para_end={sentences[-1][1]})")

In [ ]:
# Write sentence lists to text files for inspection
# Format: one sentence per line, with [PARA_END] marker on paragraph-ending sentences
for lang in LANGUAGES:
    out_path = input_dir / f"{STEM}_sentences_{lang}.txt"
    lines = []
    for sentence, is_para_end in sentence_lists[lang]:
        marker = "  [PARA_END]" if is_para_end else ""
        lines.append(f"{sentence}{marker}")
    out_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"✓ {lang}: {len(lines)} sentences → {out_path.name}")


## 2. Load Models

- **Cebuano / Tagalog**: Facebook MMS VITS models (~140 MB each, native phonemes)
- **English**: Coqui XTTS v2 (`tts_models/multilingual/multi-dataset/xtts_v2`, speaker: Damien Black)
  — higher quality than MMS for English prose; preserves capitalisation and punctuation.

MMS models are cached by HuggingFace after first download. XTTS v2 is cached by Coqui TTS.


In [ ]:
print("Loading models...")
t0 = time.time()

models = {}
tokenizers = {}

# Load MMS VITS models for Cebuano and Tagalog
for lang in ["ceb", "tl"]:
    model_id = MMS_MODELS[lang]
    lang_label = {"ceb": "Cebuano", "tl": "Tagalog"}[lang]
    print(f"  Loading {lang_label} ({model_id})...")
    t_lang = time.time()
    tokenizers[lang] = AutoTokenizer.from_pretrained(model_id)
    models[lang] = VitsModel.from_pretrained(model_id)
    models[lang].eval()
    print(f"  ✓ {lang_label} loaded in {time.time() - t_lang:.1f}s")

# Load Coqui XTTS v2 for English
print(f"  Loading English (Coqui XTTS v2)...")
t_lang = time.time()
os.environ["COQUI_TOS_AGREED"] = "1"
from TTS.api import TTS
device = "cuda" if torch.cuda.is_available() else "cpu"
xtts_engine = TTS(XTTS_MODEL_NAME).to(device)
models["en"] = xtts_engine
print(f"  ✓ English (XTTS v2) loaded in {time.time() - t_lang:.1f}s on {device}")

print(f"\n✓ All models loaded in {time.time() - t0:.1f}s total")


## 3. Synthesis

`synthesize_with_mms` handles Cebuano and Tagalog:
1. Tokenize → VITS model inference → float32 waveform
2. Convert to int16 PCM → pydub AudioSegment
3. Append silence: 150ms between sentences, 150ms after paragraph-final sentences
4. Apply speed multiplier (1.1×) → concatenate → export MP3 at 128kbps

`synthesize_with_xtts` handles English:
1. Pass sentence to Coqui XTTS v2 → 24kHz float32 waveform
2. Convert to int16 PCM → pydub AudioSegment
3. Append silence: 300ms between sentences, 500ms after paragraph-final sentences
4. Concatenate → export MP3 at 128kbps

Output: `notebooks/08-mms-tts-experiment/{stem}_tts_{lang}.mp3`


In [ ]:
def speed_change(segment: AudioSegment, speed: float) -> AudioSegment:
    """Speed up audio by resampling (exact speedup, trivial pitch shift at modest rates)."""
    new_rate = int(segment.frame_rate * speed)
    return segment._spawn(segment.raw_data, overrides={"frame_rate": new_rate}).set_frame_rate(segment.frame_rate)


def synthesize_with_mms(
    sentences: list[tuple[str, bool]],
    model,
    tokenizer,
    output_path: Path,
    sentence_pause_ms: int = 150,
    paragraph_pause_ms: int = 150,
    speech_speed: float = 1.0,
) -> Path:
    """Synthesize MMS VITS sentence list to MP3 (Cebuano / Tagalog)."""
    if not sentences:
        raise ValueError("sentences list is empty")
    rate = model.config.sampling_rate
    combined = AudioSegment.empty()
    for sentence, is_paragraph_end in sentences:
        if not sentence.strip():
            continue
        inputs = tokenizer(sentence, return_tensors="pt")
        with torch.no_grad():
            waveform = model(**inputs).waveform
        pcm = (waveform.squeeze().numpy() * 32_767).clip(-32_768, 32_767).astype(np.int16)
        segment = AudioSegment(pcm.tobytes(), frame_rate=rate, sample_width=2, channels=1)
        if speech_speed != 1.0:
            segment = speed_change(segment, speech_speed)
        combined += segment
        pause_ms = paragraph_pause_ms if is_paragraph_end else sentence_pause_ms
        combined += AudioSegment.silent(duration=pause_ms, frame_rate=rate)
    combined.export(str(output_path), format="mp3", bitrate="128k")
    return output_path


def synthesize_with_xtts(
    sentences: list[tuple[str, bool]],
    tts_engine,
    output_path: Path,
    speaker: str = XTTS_SPEAKER,
    sentence_pause_ms: int = 300,
    paragraph_pause_ms: int = 500,
) -> Path:
    """Synthesize Coqui XTTS v2 sentence list to MP3 (English)."""
    if not sentences:
        raise ValueError("sentences list is empty")
    combined = AudioSegment.empty()
    for sentence, is_paragraph_end in sentences:
        if not sentence.strip():
            continue
        wav = tts_engine.tts(text=sentence, speaker=speaker, language="en")
        pcm = (np.array(wav) * 32_767).clip(-32_768, 32_767).astype(np.int16)
        segment = AudioSegment(pcm.tobytes(), frame_rate=XTTS_SAMPLE_RATE, sample_width=2, channels=1)
        combined += segment
        pause_ms = paragraph_pause_ms if is_paragraph_end else sentence_pause_ms
        combined += AudioSegment.silent(duration=pause_ms, frame_rate=XTTS_SAMPLE_RATE)
    combined.export(str(output_path), format="mp3", bitrate="128k")
    return output_path


# --- Run synthesis: all three languages ---
synthesis_config = {
    "ceb": {"sentence_pause_ms": 150, "paragraph_pause_ms": 150, "speech_speed": 1.1},
    "tl":  {"sentence_pause_ms": 150, "paragraph_pause_ms": 150, "speech_speed": 1.1},
    "en":  {"sentence_pause_ms": 300, "paragraph_pause_ms": 500},
}

results = {}

for lang in LANGUAGES:
    lang_label = {"ceb": "Cebuano", "tl": "Tagalog", "en": "English"}[lang]
    output_path = output_dir / f"{STEM}_tts_{lang}.mp3"
    sentences = sentence_lists[lang]
    config = synthesis_config[lang]
    print(f"Synthesizing {lang_label} ({len(sentences)} sentences)...")
    t_start = time.time()
    print(sentences)

    if lang == "en":
        synthesize_with_xtts(
            sentences,
            models["en"],
            output_path,
            sentence_pause_ms=config["sentence_pause_ms"],
            paragraph_pause_ms=config["paragraph_pause_ms"],
        )
    else:
        synthesize_with_mms(
            sentences,
            models[lang],
            tokenizers[lang],
            output_path,
            sentence_pause_ms=config["sentence_pause_ms"],
            paragraph_pause_ms=config["paragraph_pause_ms"],
            speech_speed=config["speech_speed"],
        )

    elapsed = time.time() - t_start
    size_kb = output_path.stat().st_size // 1024
    duration_s = (size_kb * 1024 * 8) / 128_000
    print(f"  ✓ {elapsed:.1f}s  |  {size_kb} KB  |  ~{duration_s:.0f}s audio")
    print(f"  ✓ {output_path.name}")

    results[lang] = {
        "lang_label": lang_label,
        "elapsed_s": round(elapsed, 1),
        "size_kb": size_kb,
        "duration_s": round(duration_s),
        "path": output_path,
    }

print("\n✓ Synthesis complete")


## 4. Manual Audio Assessment

Listen to each MP3 using the players below, then fill in your scores in the
assessment dict in the next cell.

**Score guide:**
- `quality_score` (1–5): Overall audio quality and clarity
- `natural_filipino` (1–5): How natural the Filipino pronunciation sounds (CEB/TL only;
  use for EN to rate overall naturalness)

Run the cell after filling in your scores — the comparison table in the next section
uses these values.

In [ ]:
from IPython.display import Audio, display

for lang in LANGUAGES:
    lang_label = results[lang]["lang_label"]
    print(f"--- {lang_label} ---")
    display(Audio(str(results[lang]["path"]), autoplay=False))
    print()

In [ ]:
# ── FILL IN YOUR SCORES AFTER LISTENING ──────────────────────────────────────
assessment = {
    "ceb": {
        "quality_score": None,     # 1–5: overall audio quality
        "natural_filipino": None,  # 1–5: naturalness of Cebuano pronunciation
        "notes": "",
    },
    "tl": {
        "quality_score": None,
        "natural_filipino": None,  # 1–5: naturalness of Tagalog pronunciation
        "notes": "",
    },
    "en": {
        "quality_score": None,
        "natural_filipino": None,  # 1–5: naturalness / clarity
        "notes": "",
    },
}
# ─────────────────────────────────────────────────────────────────────────────

print("Assessment recorded. Run the next cell to see the comparison table.")
for lang, scores in assessment.items():
    label = {"ceb": "Cebuano", "tl": "Tagalog", "en": "English"}[lang]
    q = scores["quality_score"] or "—"
    n = scores["natural_filipino"] or "—"
    print(f"  {label}: quality={q}/5  naturalness={n}/5  notes={scores['notes']!r}")

## 5. Comparison: MMS TTS vs Coqui XTTS v2

### XTTS v2 baseline (notebook 07, CEB only)
| Metric | XTTS v2 (CEB) |
|---|---|
| Synthesis time | 821s |
| Audio duration | ~467s (~7.8 min) |
| Language mapping | Spanish phoneme approximation |
| Model size | 1.87 GB |

### English TTS options evaluated

| Model | Quality | Notes |
|---|---|---|
| `facebook/mms-tts-eng` | ❌ Poor | Robotic, unnatural prosody — rejected |
| `microsoft/speecht5_tts` + HiFiGAN | ❌ Poor | Scratchy output, random speaker embeddings gave inconsistent voice — rejected |
| Coqui XTTS v2 (`Damien Black`) | ✅ Good | Natural, authoritative broadcast tone — **selected** |

### MMS TTS (CEB/TL) + Coqui XTTS v2 (EN)
| Language | Synthesizer | Approx. synthesis time | Notes |
|---|---|---|---|
| Cebuano | MMS VITS | ~42s | Native phonemes, ~20× faster than XTTS v2 |
| Tagalog | MMS VITS | ~36s | Native phonemes |
| English | Coqui XTTS v2 | ~60s (GPU) | Best English quality |

### Production decision
**MMS (CEB/TL) + Coqui XTTS v2 (EN)** is the production configuration,
implemented in `modal_etl/step3_tts.py` (`SYNTHESIZER_MAP`).


In [ ]:
# XTTS v2 baseline from notebook 07 (CEB only)
xtts_baseline = {
    "ceb": {"elapsed_s": 821.2, "size_kb": 7293, "duration_s": 467,
            "quality_score": None, "natural_filipino": None,
            "phoneme": "es (Spanish approx)", "model_size": "1.87 GB"},
}

MODEL_LABELS = {"ceb": "MMS VITS", "tl": "MMS VITS", "en": "Coqui XTTS v2"}

print("MMS TTS vs XTTS v2 — PAGASA_20-19W_Pepito_SWB#01")
print("=" * 90)
print(f"{'Model':<12} {'Lang':<10} {'Time':>7} {'Audio':>7} {'Size':>8}  "
      f"{'Quality':>8} {'Naturalness':>12}  {'Phoneme'}")
print("-" * 90)

for lang in LANGUAGES:
    lang_label = results[lang]["lang_label"]
    r = results[lang]
    a = assessment[lang]
    phoneme = "native" if lang in ("ceb", "tl") else "native (en)"
    q = f"{a['quality_score']}/5" if a["quality_score"] else "—"
    n = f"{a['natural_filipino']}/5" if a["natural_filipino"] else "—"
    print(f"{MODEL_LABELS[lang]:<12} {lang_label:<10} {r['elapsed_s']:>6.1f}s {r['duration_s']:>6}s "
          f"{r['size_kb']:>7}KB  {q:>8} {n:>12}  {phoneme}")

print()
for lang in ["ceb"]:
    b = xtts_baseline[lang]
    lang_label = "Cebuano"
    q = f"{b['quality_score']}/5" if b["quality_score"] else "—"
    n = f"{b['natural_filipino']}/5" if b["natural_filipino"] else "—"
    print(f"{'XTTS v2':<12} {lang_label:<10} {b['elapsed_s']:>6.1f}s {b['duration_s']:>6}s "
          f"{b['size_kb']:>7}KB  {q:>8} {n:>12}  {b['phoneme']}")

print("=" * 90)
print(f"\nModel footprint: MMS CEB+TL ~280 MB + XTTS v2 ~1.87 GB = ~2.15 GB total")
print(f"vs XTTS v2: 1,870 MB")

# Speedup for CEB (the one language with an XTTS v2 baseline)
if "ceb" in results and results["ceb"]["elapsed_s"] > 0:
    speedup = xtts_baseline["ceb"]["elapsed_s"] / results["ceb"]["elapsed_s"]
    print(f"MMS CEB speedup vs XTTS v2: {speedup:.1f}×")

## Conclusion

### English TTS — journey

Three models were tried for English before landing on XTTS v2:

1. **`facebook/mms-tts-eng`** — tried first as it would keep the stack uniform.
   Quality was poor: robotic delivery, unnatural prosody. Rejected.

2. **Microsoft SpeechT5** (`microsoft/speecht5_tts` + HiFiGAN vocoder) — tried as a
   lightweight alternative. Output was scratchy and the random speaker embeddings
   gave an inconsistent voice identity. Rejected.

3. **Coqui XTTS v2** (`Damien Black` voice) — natural, authoritative broadcast tone.
   Clearly the best option despite the larger model footprint. **Selected.**

### Final production configuration

| Language | Model | Rationale |
|---|---|---|
| Cebuano | `facebook/mms-tts-ceb` (MMS VITS) | Native phonemes, ~20× faster than XTTS v2 |
| Tagalog | `facebook/mms-tts-tgl` (MMS VITS) | Native phonemes |
| English | Coqui XTTS v2 (`Damien Black`) | Best quality; MMS eng and SpeechT5 both rejected |

MMS replaces XTTS v2 for CEB and TL — native phoneme support is a meaningful
improvement over the Spanish phoneme approximation used previously, and synthesis
is significantly faster. XTTS v2 is retained for English where it clearly outperforms
both MMS and SpeechT5.

Deployed in `modal_etl/step3_tts.py`.
